<a href="https://colab.research.google.com/github/io-uty/pyspark-data-analysis/blob/main/movie_genre_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from collections import Counter

In [ ]:
ls

sample_data/


In [ ]:
pwd

'/content'

In [ ]:
!apt-get install openjdk-8-jdk-headless -qq > /dev/null
!wget -q http://archive.apache.org/dist/spark/spark-3.5.1/spark-3.5.1-bin-hadoop3.tgz
!tar xf spark-3.5.1-bin-hadoop3.tgz
!pip install -q findspark

In [ ]:
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
os.environ["SPARK_HOME"] = "/content/spark-3.5.1-bin-hadoop3"

In [ ]:
!ls

sample_data  spark-3.5.1-bin-hadoop3  spark-3.5.1-bin-hadoop3.tgz


In [ ]:
import findspark
findspark.init()
from pyspark.sql import SparkSession
spark = SparkSession.builder.master("local[*]").getOrCreate()
spark.conf.set("spark.sql.repl.eagerEval.enabled", True) # Property used to format output tables better
spark

In [ ]:
# Downloading and preprocessing Cars Data downloaded origianlly from https://perso.telecom-paristech.fr/eagan/class/igr204/datasets
!wget https://jacobceles.github.io/knowledge_repo/colab_and_pyspark/cars.csv

--2024-07-16 07:36:46--  https://jacobceles.github.io/knowledge_repo/colab_and_pyspark/cars.csv
Resolving jacobceles.github.io (jacobceles.github.io)... 185.199.108.153, 185.199.109.153, 185.199.110.153, ...
Connecting to jacobceles.github.io (jacobceles.github.io)|185.199.108.153|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://jacobcelestine.com/knowledge_repo/colab_and_pyspark/cars.csv [following]
--2024-07-16 07:36:46--  https://jacobcelestine.com/knowledge_repo/colab_and_pyspark/cars.csv
Resolving jacobcelestine.com (jacobcelestine.com)... 185.199.108.153, 185.199.109.153, 185.199.110.153, ...
Connecting to jacobcelestine.com (jacobcelestine.com)|185.199.108.153|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 22608 (22K) [text/csv]
Saving to: ‘cars.csv’

cars.csv            100%[===================>]  22.08K  --.-KB/s    in 0.001s  

2024-07-16 07:36:46 (16.0 MB/s) - ‘cars.csv’ saved [22608/22608]



In [ ]:
dfRate = spark.read.csv('ratings.csv', header=True, sep=",")
dfRate.show()

+--------+------+-------+------+----------------+
|ratingId|userId|movieId|rating|       timestamp|
+--------+------+-------+------+----------------+
|       1|   196|    242|     3|1997-12-04 15:55|
|       2|   186|    302|     3|1998-04-04 19:22|
|       3|    22|    377|     1| 1997-11-07 7:18|
|       4|   244|     51|     2| 1997-11-27 5:02|
|       5|   166|    346|     1| 1998-02-02 5:33|
|       6|   298|    474|     4|1998-01-07 14:20|
|       7|   115|    265|     2|1997-12-03 17:51|
|       8|   253|    465|     5|1998-04-03 18:34|
|       9|   305|    451|     3| 1998-02-01 9:20|
|      10|     6|     86|     3|1997-12-31 21:16|
|      11|    62|    257|     2|1997-11-12 22:07|
|      12|   286|   1014|     5|1997-11-17 15:38|
|      13|   200|    222|     5| 1997-10-05 9:05|
|      14|   210|     40|     3|1998-03-27 21:59|
|      15|   224|     29|     3|1998-02-21 23:40|
|      16|   303|    785|     3| 1997-11-14 5:28|
|      17|   122|    387|     5|1997-11-11 17:47|


In [ ]:
dfGenres = spark.read.csv('movie_genres.csv', header = True, sep = ",")
dfGenres.show()

+--------+-------+-------+
|mgenreId|movieId|  genre|
+--------+-------+-------+
|       1|    267|unknown|
|       2|   1373|unknown|
|       3|      2| Action|
|       4|      4| Action|
|       5|     17| Action|
|       6|     21| Action|
|       7|     22| Action|
|       8|     24| Action|
|       9|     27| Action|
|      10|     28| Action|
|      11|     29| Action|
|      12|     33| Action|
|      13|     39| Action|
|      14|     50| Action|
|      15|     53| Action|
|      16|     54| Action|
|      17|     62| Action|
|      18|     68| Action|
|      19|     73| Action|
|      20|     74| Action|
+--------+-------+-------+
only showing top 20 rows



In [ ]:
dfGenres = dfGenres.dropDuplicates(['movieId', 'genre'])
dfGenres.orderBy('genre').show()
dfGenres.show()

+--------+-------+------+
|mgenreId|movieId| genre|
+--------+-------+------+
|     203|   1138|Action|
|     218|   1303|Action|
|     204|   1139|Action|
|      26|    101|Action|
|     205|   1161|Action|
|     192|   1016|Action|
|      28|    117|Action|
|     195|   1027|Action|
|      29|    118|Action|
|     197|   1076|Action|
|     206|   1180|Action|
|     199|   1088|Action|
|     207|   1181|Action|
|      27|    110|Action|
|     208|   1183|Action|
|     202|   1110|Action|
|     209|   1188|Action|
|     211|   1222|Action|
|      32|    128|Action|
|     212|   1228|Action|
+--------+-------+------+
only showing top 20 rows

+--------+-------+----------+
|mgenreId|movieId|     genre|
+--------+-------+----------+
|     389|      1| Animation|
|     431|      1|Children's|
|     553|      1|    Comedy|
|    1223|     10|     Drama|
|    2796|     10|       War|
|    1070|    100|     Crime|
|    1264|    100|     Drama|
|    2568|    100|  Thriller|
|     855|   1000|  